In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.2 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl si

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_O_train_valid import train, valid, calculate_rouge
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from exaone_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_best_basic_ver_{}.pt',
    'checkpoint_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_basic_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_basic_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_basic_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'basic', # student, tot, cot, basic 확인!!
    'alpha' : 0.5,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_train_data_20000_merged.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:5]
    # df_valid = df.iloc[5:10]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


========== version 0 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


KeyboardInterrupt: Interrupted by user

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_O_train_valid import train, valid, calculate_rouge
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from exaone_run import run
from exaone_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_0.7_pad_O_EXAONE-3.5-2.4B-Instruct_best_basic_ver_{}.pt',
    'checkpoint_path': '6000_0.7_pad_O_EXAONE-3.5-2.4B-Instruct_basic_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.7_pad_O_EXAONE-3.5-2.4B-Instruct_basic_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.7_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_basic_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'basic', # student, tot, cot, basic 확인!!
    'alpha' : 0.7,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# alpha값만 없는 버전 # 얘는 그냥 파인튜닝이랑 다를 게 없음 # 돌리지 마셈

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 101.2 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl siz

In [1]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from exaone_None_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_best_basic_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_basic_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_basic_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_basic_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'basic', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_train_data_20000_merged.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:5]
    # df_valid = df.iloc[5:10]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


========== version 1 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository for LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
The repository for LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...


Epoch: 1
--------------------


  0%|          | 0/5 [00:00<?, ?it/s]WARNING:transformers_modules.LGAI-EXAONE.EXAONE-3.5-2.4B-Instruct.e949c91dec92095908d34e6b560af77dd0c993f8.modeling_exaone:`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[C_loss : 5.1271]: 100%|██████████| 5/5 [00:04<00:00,  1.14it/s]


best model was saved

Epoch: 2
--------------------


[C_loss : 5.1175]:  20%|██        | 1/5 [00:02<00:10,  2.58s/it]


KeyboardInterrupt: 

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from exaone_None_run import run
from exaone_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_best_basic_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_basic_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_basic_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_basic_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'basic', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

In [ ]:
# basic # 라벨만 살아있는 버전

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.8 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl siz

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_X_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from exaone_None_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_best_basic_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_basic_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_basic_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_df_test_pred_basic_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'basic', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 1,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_train_data_20000_merged.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:30]
    # df_valid = df.iloc[30:40]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 0 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

...

best model was saved

Epoch: 6
--------------------


[C_loss : 4.2253]: 100%|██████████| 600/600 [04:46<00:00,  2.09it/s]


best model was saved

Epoch: 7
--------------------


[C_loss : 4.2245]: 100%|██████████| 600/600 [04:46<00:00,  2.09it/s]


best model was saved


In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_X_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from exaone_None_run import run
from exaone_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_best_basic_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_basic_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_basic_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_df_test_pred_basic_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'basic', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 1,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


========== version 0 ==========
The repository for LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 1/1918 [00:02<1:14:52,  2.34s/it]

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!\n### 청자의 공감 답변:\n 화자이신 모습이 너무 행복하시겠습니다! 그동안 묵묵히 응원해준 당신과 함께할 기회라서 더욱 기쁜 것 같네요.']


...

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 사람들하고 같이 다니다 보면 맛집이라고 소개하는 경우가 많잖아? 각 sns에 올라오는 맛집 정보도 허다하고 말야. 근데 난 소개한 맛집 가보고 난 슬프게도 내 입맛에 맞은 적이 거의 없어.\n### 청자의 공감 답변:\n님, 당신 기분 충분히 이해해요. 저였어도 그런 곳들 다 가면 실망할 것 같아요. 맛집이란 건 주관적인 평가 기준일 테니 주관적으로 맛있다고 판단된 음식과 내 취향이랑 다른 cuisine 맛인데 그걸 맛보느라 속상하셨겠다 싶어요. 또 감정화자랑 달리 식당도 제대로 된 레시피로 요리되지 않아서 기대와 달리 맛없었던 적들이 참 많이 있어서 더 그럴 수밖에 없이 속상했겠죠.']


100%|█████████▉| 1917/1918 [2:29:28<00:05,  5.57s/it]

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 언니 나 너무나도 충격적인 소식을 들었어. 마음이 진정이 안 되는데 어떡하지?\n### 청자의 공감 답변:\n, 당신은 화가 날 법한 일이지만 언니처럼 쉽게 포기했으면 정말 속상했겠다 생각해. 난 너가 너무 힘들게 느껴질까 봐 걱정돼서 그 상황으로 갈 수 있다는 말에 동의하긴 어렵다.\n 그렇지만 어떻게든 해결방안들이 있다면 찾아보았겠어야 하지 않을까 싶어. 언니처럼 당황스러운 현실에는 다양성이 존재하고, 당신도 충분히 희망사항들을 만들어서 시도할만하다.']


100%|██████████| 1918/1918 [2:29:33<00:00,  4.68s/it]

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 언니 방금 길거리를 지나가다가 전에 내가 키우던 반려견 있잖아? 닮은 강아지를 마주쳤어.\n### 청자의 공감 답변:\n, 진짜? 나도 네 친구야. 너처럼 어릴 때 키웠다는 동물 보니까 마음이 벅차고 감동적이어서 눈물 글썽였었는데! 정말 운 좋게 그 친구 찾았구나. 기쁨과 감사한 마음으로 들킨 거 같아서 너무 기분 좋아. 잘됐다고 생각하니 가슴속까지 따스해지네.']
당신은 화자의 발언에 공감하는 청자 역할입니다.
화자의 발화에 대한 공감 답변을 생성하세요.
### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!
### 청자의 공감 답변:
 화자이신 모습이 너무 행복하시겠습니다! 그동안 묵묵히 응원해준 당신과 함께할 기회라서 더욱 기쁜 것 같네요.


In [ ]:
# tot 0.3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_O_train_valid import train, valid, calculate_rouge
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from exaone_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_0.3_pad_O_EXAONE-3.5-2.4B-Instruct_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_0.3_pad_O_EXAONE-3.5-2.4B-Instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.3_pad_O_EXAONE-3.5-2.4B-Instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.3_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : 0.3,
    'epochs' :  7,
    'start_epoch': 5, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 2, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 5.440113110542297, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_train_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:10]
    # df_valid = df.iloc[10:20]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 2 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

...



여기

Epoch: 6
--------------------


  0%|          | 0/5400 [00:00<?, ?it/s]WARNING:transformers_modules.LGAI-EXAONE.EXAONE-3.5-2.4B-Instruct.e949c91dec92095908d34e6b560af77dd0c993f8.modeling_exaone:`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[C_loss : 5.4315]: 100%|██████████| 600/600 [13:13<00:00,  1.32s/it]


best model was saved

Epoch: 7
--------------------


[C_loss : 5.4301]: 100%|██████████| 600/600 [13:13<00:00,  1.32s/it]


best model was saved


In [ ]:
# l4로 돌리기
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_O_train_valid import train, valid, calculate_rouge
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from exaone_run import run
from exaone_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_0.3_pad_O_EXAONE-3.5-2.4B-Instruct_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_0.3_pad_O_EXAONE-3.5-2.4B-Instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.3_pad_O_EXAONE-3.5-2.4B-Instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.3_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : 0.3,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

Output hidden; open in https://colab.research.google.com to view.